# Preprocessing for Curb Data Real World

In [1]:
# Enable autoload for just updated files
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
import random
sys.path.append('../../../')   # Add parent directory to Python path
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *
from utils.automatisierung import *

## 1S Segmentation of 30hz

In [3]:
# Save paths for each participant
files = {
    '../../../data/Real_world_cycling/AB - 08.07/Accelerometer_labeled_9_class.csv',
#     '../../../data/Real_world_cycling/IK - 26.02.25/Accelerometer_labeled_9_class.csv',
#     '../../../data/Real_world_cycling/JG - 03.07/Accelerometer_labeled_9_class.csv',
#     '../../../data/Real_world_cycling/KF - 04.07/Accelerometer_labeled_9_class.csv',
#     '../../../data/Real_world_cycling/LB - 18.06/Accelerometer_labeled_9_class.csv',
#     '../../../data/Real_world_cycling/LCL - 03.07/Accelerometer_labeled_9_class.csv'
 }

### Segmentation just curb_scene(0/1)

In [ ]:
# Define segmentation parameters
window_size = 30  # 1 second at 30Hz
overlap = 50       # 50% overlap
channels = ['Acc-X', 'Acc-Y', 'Acc-Z']
scene_col = 'curb_scene'  # Column name containing scene information
label_col = 'curb_activity'  # Majority vote label column

# Process each file
for file_path in files:    
    # Load the data
    df = pd.read_csv(file_path)
    # Ensure NTP is in datetime format
    df['NTP'] = pd.to_datetime(df['NTP'])
    
    # For each scene (e.g., curb_scene == 0 or 1)
    for scene in [0, 1]:        
        # Filter the dataframe for the current scene
        df_scene = df[df[scene_col] == scene]        
        # Segment the data into overlapping windows
        segments = segment_acceleration_data_overlapping_numpy(
            df_scene, 
            window_size=window_size, 
            overlap=overlap, 
            channels=channels
        )
        
        # Create output filename
        output_path = file_path.replace('.csv', f'_scene{scene}_segments_1s_{overlap}overlap.npz')
        
        # Save the segmented data as a .npz file
        np.savez(output_path, segments=segments)

In [4]:
df = pd.read_csv('../../data/Real_world_cycling/AB - 08.07/Accelerometer_labeled.csv')
print_sampling_frequency(df)

Sampling frequency: 33.56 Hz


### Segmentation with detailed label curb_activity and curb_scene(0/1)

In [4]:
# Define segmentation parameters
window_size = 30  # 1 second at 30Hz
overlap = 50       # 50% overlap
channels = ['Acc-X', 'Acc-Y', 'Acc-Z']
scene_col = 'curb_scene'  # Column name containing scene information
label_col = 'curb_activity'  # Majority vote label column

# Process each file
for file_path in files:    
    # Load the data
    df = pd.read_csv(file_path)
    # Ensure NTP is in datetime format
    df['NTP'] = pd.to_datetime(df['NTP'])
    # For each scene (e.g., curb_scene == 0 or 1)
    for scene in [0, 1]:        
        # Filter the dataframe for the current scene
        df_scene = df[df[scene_col] == scene]        
        # Segment the data and get majority labels
        segments, curb_activity = segment_acceleration_data_overlapping_numpy_with_curb_activity(
            df_scene, 
            window_size=window_size, 
            overlap=overlap, 
            channels=channels,
            label_col=label_col
        )
        # Create output filename
        output_path = file_path.replace('.csv', f'_scene{scene}_segments_1s_{overlap}overlap_with_curb_activity.npz')
        # Save the segmented data and majority labels as a .npz file
        np.savez(output_path, segments=segments, curb_activity=curb_activity)

In [6]:
import numpy as np

# Example file path (update as needed)
file_path = '../../../data/Real_world_cycling/AB - 08.07/Accelerometer_labeled_9_class_scene1_segments_1s_50overlap_with_curb_activity.npz'

# Load the .npz file
data = np.load(file_path)

# Check available arrays
print("Arrays in file:", data.files)

# Inspect shapes
print("Segments shape:", data['segments'].shape)
print("Majority labels shape:", data['curb_activity'].shape)

# Show unique values and their counts for majority_labels
unique, counts = np.unique(data['curb_activity'], return_counts=True)
print("Unique curb_activity:", unique)
print("Counts:", counts)

Arrays in file: ['segments', 'curb_activity']
Segments shape: (82, 30, 3)
Majority labels shape: (82,)
Unique curb_activity: [1 2 3 4 5 6 7]
Counts: [18  2  6 37 10  4  5]
